# CS432 Track 1 - Module B Submission Report
## BlindDrop Local API, RBAC, Audit, Integrity, and SQL Optimization


This notebook is the main evaluator-facing Module B report. It is written as a structured technical submission rather than a short checklist, and it ties together the source code, schema, API evidence, audit artifact, tamper-detection logic, and SQL optimization evidence packaged with the module.


## Video Demonstration

Hosted demo URL: https://youtu.be/FzY8OeX4d5E?si=ptfeexguYQ99MdmG




In [ ]:
from pathlib import Path
import json
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "evidence").exists():
    ROOT = Path(r"F:\SEM IV\lessons\DB\Project\CS432_Track1_Submission\Module_B")

EVIDENCE = ROOT / "evidence"
DOCS = ROOT / "docs"
LOGS = ROOT / "logs"

artifact_paths = {
    "benchmark_json": EVIDENCE / "BENCHMARK_EVIDENCE" / "07_benchmark_results.json",
    "api_summary": EVIDENCE / "API_EVIDENCE" / "12_api_evidence_summary.md",
    "api_matrix": EVIDENCE / "API_EVIDENCE" / "13_api_matrix.md",
    "audit_log": LOGS / "audit.log",
    "module_evidence": EVIDENCE / "API_EVIDENCE" / "11_module_b_evidence.txt",
}

print("Notebook root:", ROOT)
for name, path in artifact_paths.items():
    print(f"{name}: {path.exists()} -> {path}")


## Report Structure

This report is organised as a formal submission document with the following flow:

1. Problem Statement
2. Project Context and Adaptation Strategy
3. Requirement Mapping
4. Solution Overview
5. Architecture and Implementation Details
6. Schema and Data Model
7. Authentication, Session Validation, and RBAC
8. Security Controls: Audit Logging and Tamper Detection
9. API Workflow and End-to-End Evidence
10. SQL Optimization and Performance Analysis
11. Discussion of Results
12. Conclusion

That structure is broader than the minimum rubric wording on purpose. A submission report should not only state what files exist, but also explain why the design choices are technically coherent.


## 1. Problem Statement

Module B requires a local database-backed web application that demonstrates all of the following in one coherent package:

- a local UI and backend API surface
- authenticated session validation
- role-based access control with different behavior for Admin and Regular User
- CRUD over project-specific data
- local logging of sensitive actions
- identification of unauthorized direct database modification
- SQL indexing, query profiling, and measurable before/after optimization evidence

The main challenge is that these requirements should be satisfied without inventing an unrelated academic mini-app that has no connection to the actual project. The submission therefore needs to adapt the existing BlindDrop domain into an assignment-facing module while preserving technical consistency.


## 2. Project Context and Adaptation Strategy

BlindDrop is a secure temporary file-sharing system organised around expiring vaults, public outer tokens, and private inner tokens. Instead of introducing a second login system that does not belong to the product, Module B reuses the existing credential model and maps it directly into assignment roles:

- `outerToken + MAIN innerToken` => `admin`
- `outerToken + SUB innerToken` => `user`

This adaptation is important because it keeps the submission grounded in the real project domain. The assignment-facing CRUD resource is implemented through `portfolio_entries`, which provides a clean place to demonstrate:

- row ownership
- admin vs user behavior
- integrity hashing
- audit logging
- SQL optimization over a protected query path


## 3. Requirement Mapping

| Assignment requirement | Module B implementation |
| --- | --- |
| Local database-backed application | MySQL-backed BlindDrop backend and local static frontend |
| Local UI and APIs | `app/frontend/` and `app/backend/src/routes/` |
| Session validation | `/api/auth/login`, `/api/auth/isAuth`, `authSession.js` |
| Admin vs Regular User RBAC | token-type mapping from existing BlindDrop credentials |
| Project-specific CRUD | `portfolio_entries` via `/api/portfolio` |
| Local security logging | `logs/audit.log`, `fileAuditLogger.js` |
| Unauthorized DB modification detection | `portfolioIntegrity.js`, `/api/security/unauthorized-check` |
| SQL indexing and profiling | `init_schema.sql`, `schemaOptimization.js`, benchmark evidence |

The report therefore treats Module B as a coherent adaptation of the project, not as a separate toy application bolted on for the assignment.


## 4. Solution Overview

The packaged solution has four main layers.

### 4.1 Backend API layer

The backend is implemented in Node.js and Express. It exposes the assignment-facing authentication, portfolio, security, and evidence routes while still preserving the original BlindDrop workflow.

### 4.2 Database layer

MySQL stores both the original BlindDrop business tables and the assignment-facing `portfolio_entries` table used for RBAC, integrity checks, and optimization work.

### 4.3 Security and observability layer

The backend records important actions in a hash-chained audit log and verifies row integrity through a server-side hash over protected fields.

### 4.4 Optimization layer

A dedicated benchmark process captures `EXPLAIN` plans and execution timings for a protected portfolio query before and after indexing so that the submission includes measurable SQL optimization evidence rather than only index definitions.


## 5. Architecture and Implementation Details

### 5.1 Packaged architecture

- **Backend:** Node.js + Express
- **Database:** MySQL (`blinddrop_proto`)
- **Frontend:** static HTML/CSS/JavaScript served locally by Express
- **Storage for file blobs:** Google Drive in the original BlindDrop workflow
- **Assignment layer:** portfolio CRUD, RBAC, audit logging, tamper detection, and SQL optimization evidence

### 5.2 Important packaged folders

| Folder | Purpose |
| --- | --- |
| `app/backend/` | Express app, routes, services, benchmark scripts, and reports |
| `app/frontend/` | local frontend pages and scripts |
| `sql/` | submission-facing SQL copies |
| `logs/` | packaged audit artifact |
| `docs/` | final report, runbook, and technical documentation |
| `evidence/` | API, DB, audit, and benchmark evidence folders |

### 5.3 Important source files

| File | Responsibility |
| --- | --- |
| `src/routes/auth.js` | login and authenticated-session routes |
| `src/routes/portfolio.js` | assignment-facing CRUD routes |
| `src/routes/security.js` | unauthorized-check and related security routes |
| `src/services/authSession.js` | token-based local session validation |
| `src/services/portfolioIntegrity.js` | integrity hashing and tamper detection |
| `src/services/fileAuditLogger.js` | hash-chained local audit writer |
| `src/services/schemaOptimization.js` | SQL optimization helpers |
| `app/backend/sql/init_schema.sql` | packaged schema and index definitions |


## 6. Schema and Data Model

### 6.1 Core tables used by the package

| Table | Purpose |
| --- | --- |
| `vaults` | vault identity, outer token, expiry, and status |
| `inner_tokens` | MAIN/SUB token records and lookup hashes |
| `files` | file-level metadata and lifecycle state |
| `file_metadata` | display and path metadata |
| `file_key_access` | token-to-file access mapping |
| `sessions` | request-session tracking |
| `auth_attempts` | security-sensitive access attempts |
| `download_logs` | download history |
| `captcha_tracking` | CAPTCHA state |
| `expiry_jobs` | expiry worker metadata |
| `portfolio_entries` | Module B CRUD resource |

### 6.2 Why `portfolio_entries` is the assignment-facing table

`portfolio_entries` is the best CRUD target because it supports all of the following naturally:

- ownership rules through `owner_token_id`
- mutation traceability through `created_by_token_id`
- soft delete through `status`
- tamper detection through `integrity_hash`
- realistic indexing for protected listing queries

### 6.3 Important indexes visible in the package

- `idx_portfolio_vault_owner_status`
- `idx_portfolio_vault_status`
- `idx_portfolio_integrity_hash`
- `idx_inner_tokens_lookup_hash`
- `idx_auth_attempts_session_time`
- `idx_download_file_time`
- `idx_download_token`
- `idx_file_key_access_token`
- `idx_vault_expiry`
- `idx_expiry_jobs_sched`

The schema therefore supports both the functional requirements and the optimization story required by the assignment.


## 7. Authentication, Session Validation, and RBAC

### 7.1 Authentication routes

- `POST /api/auth/login`
- `GET /api/auth/isAuth`

### 7.2 Effective role model

| Role | Origin | Effective behavior |
| --- | --- | --- |
| `admin` | `MAIN` inner token | can create/read/update/delete portfolio entries in the vault, can run unauthorized-check |
| `user` | `SUB` inner token | can read and update only owned active entries, cannot create, cannot delete, cannot run unauthorized-check |

### 7.3 Session validation rules

Each protected request revalidates:

- session-token validity
- vault status
- vault expiry
- token status

This prevents stale sessions from remaining usable after expiry or revocation, which is important because the assignment explicitly asks for local session validation rather than a one-time login check.


## 8. Security Controls: Audit Logging and Tamper Detection

### 8.1 Integrity protection

Each `portfolio_entries` row stores an `integrity_hash` computed from protected fields plus a server-side secret. If someone edits a row directly in MySQL without recomputing that hash through application logic, the backend can detect the mismatch.

### 8.2 Detection modes

- **Passive protection:** tampered rows are blocked during normal portfolio reads
- **Active protection:** `GET /api/security/unauthorized-check` returns a tamper summary for an admin

The packaged evidence route reports:

- integrity status: `True`
- tampered count in the captured run: `0`
- audit total events: `18`
- audit hash-chain valid: `True`

### 8.3 Audit logging model

The packaged log contains **18 events** from **2026-03-20T21:37:13.021Z** to **2026-03-20T21:59:59.613Z**.

Observed action set in the packaged log:

- `auth.login.success`
- `portfolio.create`
- `portfolio.update`
- `portfolio.delete`
- `portfolio.read.denied`
- `security.integrity_check`
- `security.unauthorized-check`
- `test.init`

Severity distribution in the packaged log:

- `INFO`: `17`
- `WARN`: `1`

Each JSON-lines entry stores `previousHash` and `entryHash`, which makes the log tamper-evident instead of being a plain append-only text file.


In [ ]:
import json
from itertools import islice

audit_path = LOGS / "audit.log"
rows = [json.loads(line) for line in audit_path.read_text(encoding="utf-8").splitlines() if line.strip()]
print("Total audit rows:", len(rows))
print("First 3 audit actions:")
for row in islice(rows, 3):
    print({"ts": row["ts"], "action": row["action"], "severity": row["severity"]})
print("Last 3 audit actions:")
for row in rows[-3:]:
    print({"ts": row["ts"], "action": row["action"], "severity": row["severity"]})


## 9. API Workflow and End-to-End Evidence

The packaged API evidence is strong because it includes both positive and negative cases rather than success-only screenshots.

Captured run context from the packaged evidence:

- Base URL: `http://localhost:4000`
- Demo outer token: `OUTERDEMO7`
- Admin role resolved as: `admin`
- User role resolved as: `user`
- Created portfolio entry ID: `9b889139-d22e-484e-8db3-6b2afdd2409f`
- Denied-action proof captured: `True`

### API matrix summary

| Workflow step | What it proves |
| --- | --- |
| Admin login | valid local session creation |
| `isAuth` | session validation and role resolution |
| Admin portfolio list/create/get/update/delete | positive CRUD behavior |
| User denied action | negative authorization behavior |
| Unauthorized-check route | active tamper-check path |
| Module B evidence route | packaged examiner-facing summary |

### Verified outcomes from the packaged run

| Check | Result |
| --- | --- |
| Admin login successful | `True` |
| User login successful | `True` |
| User denied-action proof captured | `True` |
| Unauthorized tampered count during this run | `0` |
| Module B evidence route captured | `True` |
| Reported portfolio index rows | `10` |
| Reported audit event count | `18` |
| Audit hash chain valid | `True` |

This combination is stronger than a CRUD-only demo because it proves the system both allows valid actions and blocks forbidden ones.


## 10. SQL Optimization and Performance Analysis

### 10.1 Optimization target

The protected portfolio listing query combines vault scoping, owner scoping, active-row filtering, and recent-first ordering. That makes it a good candidate for demonstrating the effect of well-ordered composite indexes.

Protected query pattern:

```sql
SELECT benchmark_id, title, updated_at
FROM portfolio_benchmark_entries
WHERE vault_id = ?
  AND owner_token_id = ?
  AND status = 'ACTIVE'
ORDER BY updated_at DESC
LIMIT 25;
```

### 10.2 Benchmark stages captured in the package

| Stage | Duration (ms) | Plan type | Key used | Rows | Extra |
| --- | ---: | --- | --- | ---: | --- |
| Baseline full scan | 452.8318 | `ALL` | `none` | 4999 | `Using where; Using filesort` |
| Composite lookup index | 40.0727 | `ref` | `idx_portfolio_benchmark_lookup` | 1 | `Backward index scan` |
| Composite + covering comparison stage | 36.8205 | `ref` | `idx_portfolio_benchmark_lookup` | 1 | `Backward index scan` |

### 10.3 Interpreting the benchmark

- baseline full scan -> composite lookup index: **11.30x faster**
- baseline full scan -> composite + covering comparison stage: **12.30x faster**
- rows examined drop from **4999** to **1** in the captured plan
- in the third stage, MySQL still selected `idx_portfolio_benchmark_lookup` rather than switching to the covering index

That last point is important. The third stage should be described as a **comparison stage with the covering index present**, not as proof that the optimizer chose the covering index.

### 10.4 Why the index order makes sense

The composite lookup index aligns with the real filter and ordering pattern of the protected query:

- `vault_id`
- `owner_token_id`
- `status`
- `updated_at`

This is why the post-index plan changes from a full scan with filesort to an indexed reference lookup with backward index scan.


In [ ]:
benchmark_images = [
    EVIDENCE / "BENCHMARK_EVIDENCE" / "03_duration_comparison.png",
    EVIDENCE / "BENCHMARK_EVIDENCE" / "04_speedup_comparison.png",
    EVIDENCE / "BENCHMARK_EVIDENCE" / "05_rows_examined.png",
]

for image_path in benchmark_images:
    display(Markdown(f"### {image_path.name}"))
    if image_path.exists():
        display(Image(filename=str(image_path)))
    else:
        print(f"Missing: {image_path}")


## 11. Discussion of Results

The packaged results support several conclusions.

### 11.1 Main findings

- the Module B adaptation remains faithful to the original BlindDrop security model rather than inventing an unrelated user system
- RBAC is enforced through real route behavior, not just documented in prose
- the audit layer records both normal and security-sensitive actions and makes the log tamper-evident through hash chaining
- tamper detection is implemented at the row level using an integrity hash and exposed through both passive and active checks
- the SQL optimization layer produces measurable before/after evidence with `EXPLAIN` output and plots

### 11.2 Why the package is defensible academically

The module is stronger than a minimal CRUD demo because it does not stop at showing success cases. It also shows:

- denied actions for the restricted role
- local evidence of auditability
- a mechanism for detecting unauthorized direct DB edits
- a measured improvement from index design rather than hand-wavy claims about performance

### 11.3 Practical interpretation

From a database perspective, the portfolio feature becomes a good assignment-facing resource because it links together relational design, authorization logic, integrity protection, and index-aware query planning in one place.


## 12. Conclusion

### 12.1 Achievements

- adapted BlindDrop into a coherent Module B submission without breaking the original domain model
- implemented local session validation and role resolution
- implemented admin/user RBAC over a project-specific CRUD resource
- packaged local audit logging with tamper-evident hash chaining
- implemented unauthorized direct database modification detection
- produced SQL optimization evidence with before/after timing and `EXPLAIN` plans

### 12.2 Findings

- the `portfolio_entries` table is a strong assignment-facing resource because it naturally supports ownership rules, integrity protection, and query optimization
- the role mapping from existing BlindDrop credentials keeps the submission more technically honest than creating a second academic login system
- the composite lookup index materially improves the protected portfolio query, reducing both latency and examined rows

### 12.3 Challenges

- the original BlindDrop product was not designed specifically around the assignment rubric, so the submission had to adapt the domain carefully rather than forcing an artificial design
- the security story needed both operational controls and explainable evidence, not only route code
- benchmark language had to be written precisely so the report did not overclaim what the optimizer actually selected in the third comparison stage

### 12.4 Future Improvements

- persist session state in a dedicated durable store instead of an in-memory service
- add automated audit-chain verification tooling as a first-class admin utility
- extend benchmark coverage to more protected queries beyond the portfolio listing path
- add richer UI affordances for surfacing tamper-detected rows and audit summaries directly in the frontend

### Final statement

`For Module B, I adapted BlindDrop into a local database-backed web application that validates sessions locally, maps existing vault credentials into admin and user roles, enforces RBAC over a project-specific portfolio resource, records sensitive actions in a tamper-evident audit log, detects unauthorized direct database edits through integrity hashes, and demonstrates measurable SQL optimization with EXPLAIN-backed benchmark evidence.`

### Hosted demo link

https://youtu.be/FzY8OeX4d5E?si=ptfeexguYQ99MdmG
